# Description

We evaluate the scalability of OPERON on the previously generated data with growing number of input variables.

In [1]:
# operon_scalability_experiment.py
#
# Runs pyoperon (Operon) on the HDF5 generated by scalability_experiment_data.py
# for multiple input dimensions d and expression kinds (R1/R2/R3).
#
# Function set requested: multiplication, square, division.
# Notes:
# - Many rational targets (your p/q polynomials) require addition/subtraction to be exactly representable.
#   This script still enforces the requested operator set.

from __future__ import annotations

import inspect
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple

import h5py
import numpy as np

from pyoperon.sklearn import SymbolicRegressor


# =========================
# Experiment configuration
# =========================

@dataclass(frozen=True)
class ExpCFG:
    # Path to the dataset generated by scalability_experiment_data.py
    h5_path: str = "scalability_experiment.h5"

    # Which expression groups to run
    kinds: Tuple[str, ...] = ("R1", "R2", "R3")

    # Seeds (repeat runs); set to (seed,) if you want a single run
    seeds: Tuple[int, ...] = (0,)

    # Operator library requested
    # (constant and variable terminals are usually implicit, but we include them explicitly when supported)
    allowed_symbols: str = "mul,div,square,constant,variable"

    # GP / search parameters (will be applied only if supported by your installed pyoperon version)
    generations: int = 1000
    population_size: int = 1000
    max_length: int = 40
    max_depth: int = 10

    # Local optimization (if supported)
    optimizer: str = "lm"               # common choices: "lm", "sgd", "lbfgs"
    optimizer_iterations: int = 50
    local_search_probability: float = 0.05
    lamarckian_probability: float = 0.5

    # Multi-objective vs single-objective selection
    # If supported, this chooses which model from final Pareto front is returned.
    model_selection_criterion: str = "mse"   # often: "mse", "aic", "bic", "mdl"

    # Output formatting
    print_best_expression: bool = True


CFG = ExpCFG()


# =========================
# Utilities
# =========================

def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    return float(np.mean((y_true - y_pred) ** 2))


def _build_operon_kwargs(cfg: ExpCFG, seed: int) -> Dict[str, Any]:
    """
    Build kwargs for SymbolicRegressor in a version-tolerant way:
    we only pass parameters that exist in the installed pyoperon.
    """
    desired: Dict[str, Any] = {
        # operator set
        "allowed_symbols": cfg.allowed_symbols,
        "symbols": cfg.allowed_symbols,             # fallback name (if used)
        "function_set": cfg.allowed_symbols,        # fallback name (if used)

        # evolution / search
        "generations": cfg.generations,
        "population_size": cfg.population_size,
        "max_length": cfg.max_length,
        "max_depth": cfg.max_depth,

        # model selection
        "model_selection_criterion": cfg.model_selection_criterion,

        # local optimization / hybrid search
        "optimizer": cfg.optimizer,
        "optimizer_iterations": cfg.optimizer_iterations,
        "local_search_probability": cfg.local_search_probability,
        "lamarckian_probability": cfg.lamarckian_probability,

        # sklearn-style seeding
        "random_state": seed,
        "seed": seed,
    }

    sig = inspect.signature(SymbolicRegressor.__init__)
    supported = set(sig.parameters.keys())

    # Prefer "allowed_symbols" if available; otherwise, keep the first matching alias.
    if "allowed_symbols" in supported:
        desired.pop("symbols", None)
        desired.pop("function_set", None)
    elif "symbols" in supported:
        desired.pop("allowed_symbols", None)
        desired.pop("function_set", None)
    elif "function_set" in supported:
        desired.pop("allowed_symbols", None)
        desired.pop("symbols", None)
    else:
        # No known parameter name for operator set; leave all three (they'll be filtered out below).
        pass

    return {k: v for k, v in desired.items() if k in supported}


def _maybe_get_expression_string(reg: SymbolicRegressor) -> str | None:
    """
    Best-effort extraction of a human-readable expression (API varies by version).
    """
    for attr in ("model_", "best_model_", "best_estimator_"):
        if hasattr(reg, attr):
            model = getattr(reg, attr)
            if model is None:
                continue
            if hasattr(reg, "get_model_string"):
                try:
                    return str(reg.get_model_string(model))
                except Exception:
                    pass

    # Some versions store Pareto front; try to stringify first entry.
    if hasattr(reg, "pareto_front_") and hasattr(reg, "get_model_string"):
        try:
            front = getattr(reg, "pareto_front_")
            if front and isinstance(front, (list, tuple)):
                cand = front[0]
                if isinstance(cand, dict) and "tree" in cand:
                    return str(reg.get_model_string(cand["tree"]))
        except Exception:
            pass

    return None


# =========================
# Main experiment
# =========================

def main() -> None:
    rows: List[Dict[str, Any]] = []

    with h5py.File(CFG.h5_path, "r") as f:
        d_list = list(map(int, f.attrs["d_list"]))
        for kind in CFG.kinds:
            for d in d_list:
                # Load dataset
                Xtr = f[f"data/{kind}/d{d}/train/X"][...].astype(np.float64, copy=False)
                ytr = f[f"data/{kind}/d{d}/train/y"][...].astype(np.float64, copy=False).reshape(-1)
                Xte = f[f"data/{kind}/d{d}/test/X"][...].astype(np.float64, copy=False)
                yte = f[f"data/{kind}/d{d}/test/y"][...].astype(np.float64, copy=False).reshape(-1)

                for seed in CFG.seeds:
                    kwargs = _build_operon_kwargs(CFG, seed=seed)
                    reg = SymbolicRegressor(**kwargs)

                    reg.fit(Xtr, ytr)

                    yhat_tr = reg.predict(Xtr)
                    yhat_te = reg.predict(Xte)

                    tr_mse = mse(ytr, yhat_tr)
                    te_mse = mse(yte, yhat_te)

                    expr = _maybe_get_expression_string(reg) if CFG.print_best_expression else None

                    row = {
                        "kind": kind,
                        "d": d,
                        "seed": seed,
                        "train_mse": tr_mse,
                        "test_mse": te_mse,
                        "expr": expr,
                    }
                    rows.append(row)

                    line = f"[{kind} | d={d:>3} | seed={seed}] train_mse={tr_mse:.6e} test_mse={te_mse:.6e}"
                    print(line)
                    if expr is not None:
                        print("  expr:", expr)

    # Optional: print a compact summary table (mean/std over seeds)
    print("\n=== Summary (mean±std over seeds) ===")
    for kind in CFG.kinds:
        for d in sorted({r["d"] for r in rows if r["kind"] == kind}):
            sub = [r for r in rows if r["kind"] == kind and r["d"] == d]
            tr = np.array([r["train_mse"] for r in sub], dtype=float)
            te = np.array([r["test_mse"] for r in sub], dtype=float)
            print(
                f"{kind} d={d:>3} | "
                f"train_mse={tr.mean():.6e}±{tr.std(ddof=0):.6e} | "
                f"test_mse={te.mean():.6e}±{te.std(ddof=0):.6e}"
            )


if __name__ == "__main__":
    main()


ImportError: dlopen(/Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/pyoperon/pyoperon.cpython-310-darwin.so, 0x0002): Library not loaded: @rpath/libz.1.dylib
  Referenced from: <C61725D1-2E7F-35B6-9299-96B0E3B85EB9> /Users/garmaev/Desktop/projects/ComplexEQL/.venv/lib/python3.10/site-packages/pyoperon/pyoperon.cpython-310-darwin.so
  Reason: tried: '/Users/runner/micromamba/envs/pyoperon/lib/libz.1.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Users/runner/micromamba/envs/pyoperon/lib/libz.1.dylib' (no such file), '/Users/runner/micromamba/envs/pyoperon/lib/libz.1.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Users/runner/micromamba/envs/pyoperon/lib/libz.1.dylib' (no such file)